# Tool Capability

> The tool-capability interface — the manage-the-tool half of the capability-unit fracture (pass-2 Thread 3)

In [ ]:
#| default_exp core.capability

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import dataclasses
import inspect
import logging
import string
import threading
import time
from abc import ABC, abstractmethod
from contextlib import contextmanager
from typing import Any, Callable, Dict, Generator, Iterator, List, Mapping, Optional, Set

from cjm_substrate.core.errors import CapabilityCancelledError, CapabilityConfigError, CapabilityInputError

# CR-4: metadata key for declarative reload triggers on config dataclass fields.
# Capability authors annotate fields whose changes require resource release:
#
#     @dataclass
#     class WhisperConfig:
#         model: str = field(default="base", metadata={RELOAD_TRIGGER: "model"})
#         temperature: float = field(default=0.0)  # no reload needed on change
#
# Capabilities then implement `_release_<trigger>()` methods (e.g. `_release_model`)
# and the substrate-provided `reconfigure_with_triggers` walks the diff between
# old and new configs, firing each affected trigger's release method.
RELOAD_TRIGGER = "reload_trigger"

_logger = logging.getLogger(__name__)

## ToolCapability

`ToolCapability` is the manage-the-tool half of the capability-unit fracture
(pass-2 Thread 3): identity, lifecycle, config, cancellation, and observability
for a tool running in a worker process. It works for both:

- **Concrete capabilities**: running in Worker processes (implement the actual logic)
- **Remote proxies**: running in the Host process (forward calls over HTTP)

What it deliberately does NOT carry: the task channel. The pre-fracture
`PluginInterface` declared an untyped `execute(*args, **kwargs)` as its one
task-shaped member; under the fracture, typed task contracts live on **task
adapters** (see `core.adapter` and the per-task `cjm-<task>-adapter-interface`
libraries), and results cross the worker boundary through the **typed wire
layer** (`core.wire`).

`PluginInterface` survives as a class-identical alias in `core.interface`
(REMOVE-AFTER-OVERHAUL: retires with the stage-8 Option C cascade + SG-48).

In [ ]:
#| export
@dataclasses.dataclass
class ConfigOption:
    """CR-11: one live option for a dynamic config field, with optional metadata."""
    value: Any                              # option value (e.g. "gemini-2.5-flash")
    label: str                              # display label (e.g. "Gemini 2.5 Flash")
    metadata: Dict[str, Any] = dataclasses.field(default_factory=dict)  # token limits, descriptions, ...


@dataclasses.dataclass
class FieldOptions:
    """CR-11: the live option domain for one dynamic config field.

    Kept SEPARATE from the static config_schema (which CR-8 hashes for drift
    detection). The capability-config UI merges these live options on top of the
    static schema; folding them into the schema would make every API capability
    perpetually 'drift'.
    """
    options: List[ConfigOption]                                            # current valid options
    constraints: Dict[str, Any] = dataclasses.field(default_factory=dict)  # derived field-constraint overrides


In [ ]:
#| export
@dataclasses.dataclass
class EnvVarSpec:
    """CR-12: one entry of a capability's spawn-time worker-environment contract.

    A capability declares the environment variables its worker subprocess reads at
    startup via `WORKER_ENV: ClassVar[List[EnvVarSpec]]`. Worker env vars are
    FIXED AT SPAWN — changing one requires a worker RESPAWN, so the substrate
    routes such changes through `reload_capability`, never `reconfigure` (the env is
    baked into the subprocess at `Popen` and can't be mutated in-process). This
    is the lifecycle distinction from a normal config field (reconfigurable in
    place via `reconfigure`/`_release_<trigger>`).

    Two flavors share this one declaration:

      - `secret=True`  : value resolved from the `SecretStore` (masked; never
                         persisted in the config store, echoed in config_schema,
                         or logged). A secret never carries a `default` — a
                         baked-in secret is a leak.
      - `secret=False` : visible value resolved from the override chain
                         (operator override > manifest `install.env_vars` >
                         this `default`); safe to display.

    Both share one injection seam: the substrate composes the resolved
    {name: value} overlay at load time and injects it into the worker env at
    spawn (extending the existing CJM_CAPABILITY_DATA_DIR / CJM_MODELS_DIR injection).
    This is "derive from behaviour, not metadata" applied to the spawn env: the
    capability declares WHICH vars it consumes + whether each is secret/required;
    the substrate owns resolution + injection.

    `options` is a forward seam for visible vars with a finite domain (e.g. a
    device selector enumerating GPUs); unused today but reserved so the
    capability-config UI / a future `set-env` surface isn't blocked.
    """
    name: str                              # The env var the worker reads, e.g. "GEMINI_API_KEY"
    secret: bool = False                   # True -> value resolved from the SecretStore, masked
    required: bool = False                 # Worker can't do useful work until this is satisfied
    label: str = ""                        # Display label for CLI / GUI affordances
    description: str = ""                  # Help text for CLI / GUI
    default: Optional[str] = None          # Visible vars only; secrets must leave this None
    options: Optional[List[str]] = None    # Forward seam: finite domain for a visible var (unused today)


In [ ]:
#| export
# Q1-A (Track 19): EnvVarSpec.default templating — `string.Template` substitution
# with a constrained placeholder vocabulary.
#
# Why templating: 7 of 8 local-model capabilities carry derived path vars
# (`HF_HOME=<CJM_MODELS_DIR>/huggingface`, `TORCH_HOME`, `NLTK_DATA`, etc.) that
# were computed at install time in each capability's `meta.py` and baked into
# `install.env_vars` as install-machine absolute paths. Moving derivation into a
# templated `EnvVarSpec.default` lets the substrate own the resolution, keeps
# the capability declaration portable across machines (SG-23), and removes the
# author footgun of "remember to put os.environ.setdefault before the ML import"
# (the lifecycle hazard Option B carried).
#
# Constraint: single-pass + non-recursive + strict-raise on unknown placeholder.
# A `${FOO}` whose name is not in the allowed vocabulary fails LOUDLY at
# `_resolve_worker_env` time (catchable as CapabilityConfigError, which multi-inherits
# ValueError per CR-5). The same check fires inside `cjm-ctl validate` so
# misconfigured templates surface at install/release rather than first
# `load_capability`.
#
# Allowed placeholders (the minimum viable set per the audit):
#   - ${CJM_MODELS_DIR}   — substrate-injected models directory (may be None)
#   - ${CJM_CAPABILITY_DATA_DIR}     — substrate-injected data directory base
#   - ${CAPABILITY_DATA_DIR}  — capability's own data subdirectory (CJM_CAPABILITY_DATA_DIR/<name>)
#   - ${CAPABILITY_NAME}      — capability name (for one-off path components like NLTK)
WORKER_ENV_TEMPLATE_PLACEHOLDERS: Set[str] = {
    "CJM_MODELS_DIR",
    "CJM_CAPABILITY_DATA_DIR",
    "CAPABILITY_DATA_DIR",
    "CAPABILITY_NAME",
}


def expand_worker_env_template(
    template: str,                       # The raw EnvVarSpec.default value (may contain ${...} placeholders)
    placeholders: Mapping[str, Optional[str]],  # Resolved values keyed by placeholder name
    *,
    capability_name: str = "",               # For error context ("template X on capability Y references ...")
    var_name: str = "",                  # For error context ("on EnvVarSpec(name=Z)")
) -> str:
    """Substitute `${VAR}` placeholders in `template` using `placeholders`.
    
    Strict mode (no `safe_substitute`): unknown placeholders raise
    `CapabilityConfigError` with descriptive context. Single-pass, non-recursive —
    substituted values are taken verbatim, never re-scanned for further
    placeholders. Templates without any `${...}` syntax pass through unchanged
    (so plain static defaults work as before).
    
    The allowed placeholder vocabulary is fixed via `WORKER_ENV_TEMPLATE_PLACEHOLDERS`.
    A `${FOO}` whose name is in the vocabulary but whose RESOLVED value is None
    (e.g. `CJM_MODELS_DIR` when the operator hasn't configured one) raises
    `CapabilityConfigError` with the same shape — operators get a clear signal that
    the capability needs a value they haven't provided, rather than a silent
    substitution of empty string into a load-bearing path.
    """
    # Fast path: nothing to substitute.
    if "$" not in template:
        return template
    # Identify placeholder names referenced in the template before substituting,
    # so we can surface a precise CapabilityConfigError for unknown names without
    # relying on string.Template's stock KeyError message.
    tmpl = string.Template(template)
    referenced: Set[str] = set()
    for match in tmpl.pattern.finditer(template):
        named = match.group("named") or match.group("braced")
        if named:
            referenced.add(named)
    # Validate against the allowed vocabulary.
    unknown = referenced - WORKER_ENV_TEMPLATE_PLACEHOLDERS
    if unknown:
        ctx = f"on capability {capability_name!r}" if capability_name else ""
        if var_name:
            ctx = f"{ctx} EnvVarSpec(name={var_name!r})" if ctx else f"EnvVarSpec(name={var_name!r})"
        raise CapabilityConfigError(
            f"unknown placeholder(s) {sorted(unknown)} in worker-env default {template!r} "
            f"{ctx}; allowed: {sorted(WORKER_ENV_TEMPLATE_PLACEHOLDERS)}",
            fields_invalid=[var_name] if var_name else None,
        )
    # Resolve. A referenced-but-unresolved placeholder (allowed vocabulary,
    # value None in `placeholders`) is a configuration error from the operator
    # side, NOT a capability author bug — distinguished error message.
    missing = [n for n in referenced if placeholders.get(n) is None]
    if missing:
        ctx = f"on capability {capability_name!r}" if capability_name else ""
        if var_name:
            ctx = f"{ctx} EnvVarSpec(name={var_name!r})" if ctx else f"EnvVarSpec(name={var_name!r})"
        raise CapabilityConfigError(
            f"worker-env default {template!r} {ctx} references placeholder(s) "
            f"{sorted(missing)} but no value is configured for them "
            f"(operator must set e.g. CJM_MODELS_DIR via cjm.yaml or environment)",
            fields_invalid=[var_name] if var_name else None,
        )
    # Strict substitution. We've pre-validated unknowns + missings, so
    # substitute() can't raise KeyError; this is defense in depth.
    return tmpl.substitute({k: v for k, v in placeholders.items() if v is not None})


def template_check_placeholders(
    template: str,                       # The raw EnvVarSpec.default value
) -> Set[str]:                            # Placeholder names referenced (allowed-vocabulary-validated)
    """Return the set of placeholder names referenced by a worker-env template.
    
    Validates the vocabulary (unknown names raise CapabilityConfigError) without
    requiring a placeholder-value mapping. Useful for `cjm-ctl validate`'s
    dry-run check at install/release time — surface the bug BEFORE the capability
    tries to spawn a worker with a malformed default.
    
    Templates without `${...}` return an empty set.
    """
    if "$" not in template:
        return set()
    tmpl = string.Template(template)
    referenced: Set[str] = set()
    for match in tmpl.pattern.finditer(template):
        named = match.group("named") or match.group("braced")
        if named:
            referenced.add(named)
    unknown = referenced - WORKER_ENV_TEMPLATE_PLACEHOLDERS
    if unknown:
        raise CapabilityConfigError(
            f"unknown placeholder(s) {sorted(unknown)} in worker-env default {template!r}; "
            f"allowed: {sorted(WORKER_ENV_TEMPLATE_PLACEHOLDERS)}",
        )
    return referenced

In [ ]:
#| export
class ToolCapability(ABC):
    """Tool-capability interface: manage the tool/worker — identity, lifecycle,
    config, cancellation, observability (pass-2 Thread 3 fracture).

    The task channel is NOT part of this surface: `execute` left the abstract
    set when the capability-unit fracture split tool capabilities from task
    adapters. Typed task contracts live on adapters (`core.adapter` + the
    per-task `cjm-<task>-adapter-interface` libraries). Fused-era capabilities (the
    pre-Option-C 12) still define `execute` themselves and their domain ABCs
    still declare it abstract — they keep working unchanged through the
    class-identical `ToolCapability` alias in `core.interface` until the
    Option C migration cascade splits them.

    CR-4 extended this surface with: prefetch hook (SG-19), made cleanup optional
    (SG-43), reconfigure split + RELOAD_TRIGGER declarative-helper (lifecycle
    hook split), and cooperative-cancellation primitives (SG-16 — flag + callback
    + context manager).

    Abstract methods: `name`, `version`, `initialize`, `get_config_schema`,
    `get_current_config`. Concrete defaults (overridable): `execute_stream`
    (transitional — see method), `cleanup`, `prefetch`, `reconfigure`,
    `cancel`, `check_cancel`, `register_cancel_callback`, `cancel_signal_to`,
    `report_progress`, `report_usage`, `fields_that_changed`,
    `reconfigure_with_triggers`, `on_disable`, `on_enable`.
    """

    # CR-4: cooperative-cancellation flag. Class-level default so subclasses
    # don't have to remember to initialize it in __init__; setting on an
    # instance creates an instance attribute that shadows the class default.
    # Reset by the substrate's worker /execute wrapper before each call so
    # stale state from a previous cancellation doesn't leak forward.
    _cancel_requested: bool = False

    @property
    @abstractmethod
    def name(self) -> str: # Unique identifier for this capability
        """Unique capability identifier."""
        ...

    @property
    @abstractmethod
    def version(self) -> str: # Semantic version string (e.g., "1.0.0")
        """Capability version."""
        ...

    @abstractmethod
    def initialize(
        self,
        config: Optional[Dict[str, Any]] = None # Configuration dictionary
    ) -> None:
        """Initialize or re-configure the capability.
        
        CR-4: this is "first-time setup" — called once after construction with
        the initial config. Substrate uses `reconfigure(old, new)` for delta
        updates afterwards. Capabilities predating CR-4 see no behavior change since
        the default `reconfigure()` body delegates to `reconfigure_with_triggers`
        which is a no-op unless the capability opts in via RELOAD_TRIGGER metadata.
        """
        ...

    def execute_stream(
        self,
        *args,
        **kwargs
    ) -> Generator[Any, None, None]: # Yields partial results
        """Stream execution results chunk by chunk.

        TRANSITIONAL(option-c-cascade): streaming is substrate/composition-
        supplied under the pass-2 fracture (off both interfaces); the default
        stays here only because fused-era capabilities rely on it (it calls the
        capability's own `execute`, which a split tool capability does not have).
        Relocates when CR-17 adapter routing lands (execution stage 4).
        """
        # Fused-era default: yield single result from execute().
        yield self.execute(*args, **kwargs)

    @abstractmethod
    def get_config_schema(self) -> Dict[str, Any]: # JSON Schema for configuration
        """Return JSON Schema describing the capability's configuration options."""
        ...

    @abstractmethod
    def get_current_config(self) -> Dict[str, Any]: # Current configuration values
        """Return the current configuration state as a dictionary."""
        ...

    def get_config_options(self) -> Dict[str, "FieldOptions"]:
        """CR-11: live option domains for dynamic config fields, keyed by field name.

        Optional. Default: {} (fully static capabilities). For fields whose valid
        domain is determined at runtime (e.g. an API model list), return a
        `FieldOptions` carrying current `ConfigOption` values + per-option
        metadata (token limits, etc.) + optional derived constraints. Runs in
        the worker subprocess (has the capability's deps + credentials).

        Kept SEPARATE from get_config_schema(): the schema is static + hashed for
        CR-8 drift detection; these options are the live, un-hashed companion the
        capability-config UI merges on top. A fetch failure should raise a typed CR-5
        error; the substrate's CapabilityManager.get_config_options accessor degrades
        to {} so the UI can fall back to the static schema.
        """
        return {}

    def cleanup(self) -> None:
        """Clean up resources when capability is unloaded.
        
        CR-4: made optional (SG-43 closure). Was `@abstractmethod` before; every
        audited capability overrode it with a near-no-op, so the default is now a
        no-op and capability authors override only when they have non-trivial
        teardown (file handles, GPU memory, database connections). The
        substrate's worker /cleanup endpoint calls this regardless.
        """
        pass

    def prefetch(self) -> None:
        """Acquire heavy resources eagerly without invoking execute().
        
        CR-4 (SG-19): default no-op. Capability authors override when downstream
        callers benefit from eager acquisition — typically transcription /
        inference capabilities that lazy-download models on first execute. The
        substrate's prefetch_capability(name) API + worker /prefetch endpoint
        invoke this. Should be idempotent (safe to call multiple times) since
        the substrate may pre-warm at load time AND on operator request.
        """
        pass

    def reconfigure(
        self,
        old_config: Optional[Dict[str, Any]],  # Previous configuration values
        new_config: Optional[Dict[str, Any]],  # New configuration values to apply
    ) -> None:
        """Apply a configuration change without re-running full initialize().
        
        CR-4 completion (2026-05-25): reconfigure is the substrate's canonical
        delta path - `CapabilityManager.update_capability_config` routes here, NOT through
        a bare `initialize(new_config)`. It fires `_release_<trigger>` releases for
        changed RELOAD_TRIGGER fields, then re-applies config (see body below).
        
        Distinction from initialize(): initialize sets up persistent state once
        after construction; reconfigure applies delta updates and is the
        canonical entry point for hot-reload via the substrate's
        update_capability_config path.
        """
        self.reconfigure_with_triggers(old_config or {}, new_config or {})
        # CR-4 completion: re-apply config + run config-derived setup. Capabilities
        # that factor config into `_apply_config(config)` get the clean
        # init-once / reconfigure-delta split; others fall back to a full
        # `initialize(new_config)` (idempotent; retained manual diff-and-reload
        # blocks are harmless - `_release_*` guards make the second release a no-op).
        apply_config = getattr(self, "_apply_config", None)
        if callable(apply_config):
            apply_config(new_config or {})
        else:
            self.initialize(new_config or {})

    def fields_that_changed(
        self,
        old: Dict[str, Any],  # Previous config snapshot
        new: Dict[str, Any],  # Proposed new config snapshot
    ) -> Set[str]:  # Field names whose values differ between old and new
        """Return the set of field names whose values differ between old and new.
        
        Includes fields present in only one dict (treated as a change to/from
        the implicit None). Equality is structural via `!=`; nested dicts /
        lists compare by value, not identity. Hashable-vs-unhashable values
        compare correctly because we never put them in a set themselves.
        """
        all_keys = set(old) | set(new)
        return {k for k in all_keys if old.get(k) != new.get(k)}

    def reconfigure_with_triggers(
        self,
        old_config: Dict[str, Any],  # Previous configuration values
        new_config: Dict[str, Any],  # New configuration values being applied
    ) -> None:
        """CR-4 helper: walk RELOAD_TRIGGER metadata + fire `_release_<trigger>` methods.
        
        Resolution sequence:
        
          1. Find the capability's `config_class` attribute (a dataclass). Absent
             means the capability hasn't opted into the declarative pattern; the
             helper returns silently.
          2. Compute the diff between `old_config` and `new_config` via
             `fields_that_changed`.
          3. For each changed field, read its `RELOAD_TRIGGER` metadata key
             (if any) and accumulate the trigger names into a set (de-duping
             when multiple fields share a trigger).
          4. For each trigger, call `self._release_<trigger>()` if it exists
             on the instance.
        
        Capability authors opt in by:
        
          - Setting `config_class = MyConfigDataclass` as a class attribute.
          - Annotating field metadata with `{RELOAD_TRIGGER: "model"}` etc.
          - Implementing matching `_release_model(self)` instance methods.
        
        Capabilities WITHOUT config_class or RELOAD_TRIGGER metadata land here as a
        no-op — safe default for the SG-T3tr migration window where the
        cascade hasn't yet adopted the declarative pattern everywhere.
        """
        config_class = getattr(type(self), "config_class", None)
        if config_class is None or not dataclasses.is_dataclass(config_class):
            return
        changed = self.fields_that_changed(old_config, new_config)
        if not changed:
            return
        triggers_to_fire: Set[str] = set()
        for f in dataclasses.fields(config_class):
            if f.name not in changed:
                continue
            trigger = f.metadata.get(RELOAD_TRIGGER) if hasattr(f, "metadata") else None
            if trigger:
                triggers_to_fire.add(trigger)
        for trigger in triggers_to_fire:
            release_method = getattr(self, f"_release_{trigger}", None)
            if callable(release_method):
                try:
                    release_method()
                except Exception as e:
                    # A misbehaving _release method must NOT prevent the rest
                    # from firing — heavy-resource teardown is best-effort.
                    _logger.warning(
                        "_release_%s() raised on %r; continuing with remaining triggers: %s",
                        trigger, getattr(self, "name", type(self).__name__), e,
                    )

    def cancel(self) -> None:
        """Request cooperative cancellation of the current execute() call.
        
        CR-4: default sets the substrate-tracked `_cancel_requested` flag and
        fires any callbacks registered via `register_cancel_callback`.
        
        Capability authors who need extra teardown logic (signaling a subprocess,
        closing a network connection) override `cancel()` and SHOULD call
        `super().cancel()` to preserve the flag-setting + callback-fire
        behavior. The capability's `execute()` polls via `check_cancel()` at safe
        interruption points and unwinds when it raises `CapabilityCancelledError`.
        """
        self._cancel_requested = True
        for cb in list(getattr(self, "_cancel_callbacks", ())):
            try:
                cb()
            except Exception as e:
                # Cancellation callbacks are best-effort; a misbehaving one
                # must not stop subsequent callbacks from firing.
                _logger.warning(
                    "cancel callback raised on %r: %s",
                    getattr(self, "name", type(self).__name__), e,
                )

    def check_cancel(self) -> None:
        """Raise `CapabilityCancelledError` if cancellation has been requested.
        
        CR-4 (SG-16 polling primitive): capability authors call this at safe
        interruption points inside `execute()`. The substrate sets the flag via
        `cancel()` (typically driven by an operator's "Cancel" button); the
        next `check_cancel` call surfaces the cancellation as a typed exception
        that unwinds execute() cleanly.
        
        The substrate's worker /execute wrapper resets the flag before each
        call so cancellation doesn't leak from one job into the next.
        """
        if self._cancel_requested:
            raise CapabilityCancelledError(self.name)

    def register_cancel_callback(
        self,
        callback: Callable[[], None],  # Called when cancel() fires
    ) -> None:
        """Register a callback that fires when cancel() is called.
        
        CR-4 (SG-16 callback primitive): for capabilities that can't easily insert
        polling at strategic points (e.g., a capability wrapping a blocking C
        extension). Callbacks should be non-blocking and idempotent. Multiple
        callbacks can be registered; all fire in registration order when
        cancel() is invoked. A misbehaving callback that raises is logged
        and skipped — the remaining callbacks still fire.
        """
        if not hasattr(self, "_cancel_callbacks"):
            self._cancel_callbacks: List[Callable[[], None]] = []
        self._cancel_callbacks.append(callback)

    @contextmanager
    def cancel_signal_to(
        self,
        callback: Callable[[], None],  # Cancellation callback scoped to the with-block
    ) -> Generator[None, None, None]:
        """Context manager registering `callback` for the duration of the with-block.
        
        Useful for binding cancellation to a finite-scope resource (a subprocess,
        a network request, a temporary file handle) without needing to
        deregister manually. Pairs with `register_cancel_callback` for cases
        where lifetime is tied to a `try:`/`finally:` block.
        
        Example:
        
            def execute(self, *args, **kwargs):
                proc = subprocess.Popen(...)
                with self.cancel_signal_to(lambda: proc.terminate()):
                    return proc.wait()
        """
        self.register_cancel_callback(callback)
        try:
            yield
        finally:
            if hasattr(self, "_cancel_callbacks"):
                try:
                    self._cancel_callbacks.remove(callback)
                except ValueError:
                    pass  # Already removed (e.g. by an aliased call); silent OK.

    def on_disable(self) -> None:
        """CR-2: signal that the substrate has marked this capability as disabled.
        
        Worker stays alive; capability can release heavy resources here (e.g., free
        GPU memory, close model files). The substrate fires this hook AFTER any
        in-flight job for this capability finishes — see CapabilityManager.disable_capability
        deferred-hook semantics. Default: no-op; capabilities opt in by overriding.
        """
        pass

    def on_enable(self) -> None:
        """CR-2: signal that the substrate has marked this capability as re-enabled.
        
        Capability can eagerly re-acquire heavy resources here, or rely on lazy
        re-acquisition via the next execute() call (substrate doesn't prefer
        one strategy over the other). Default: no-op; capabilities opt in by overriding.
        """
        pass

    def report_progress(
        self,
        progress: float, # 0.0 to 1.0, or -1.0 for indeterminate
        message: str = "" # Descriptive status message
    ) -> None:
        """Report execution progress. Call during execute() to update status."""
        # Default: store in instance variables for worker to poll
        self._progress = progress
        self._status_message = message

    def report_usage(
        self,
        usage: Dict[str, float],  # Measured usage for this execute, keyed by capability-defined unit name
    ) -> None:
        """SG-54: report measured API/service usage for the current execute() call.

        Unit-agnostic by design — the capability (which holds the API response)
        supplies whatever unit names it measures: {"input_tokens": .., "output_tokens": ..}
        for an LLM, {"pages": ..} for OCR, {"characters": ..} for TTS,
        {"credits"/"requests"/"minutes": ..} for others. The substrate stores +
        accumulates per unit name WITHOUT interpreting them (summed across runs in
        the empirical store's api_usage_totals). Pricing is deliberately NOT here
        (volatile, per-service, often not API-accessible) — a consumer-side rate
        table turns raw units into cost. "Derive from behaviour": the capability
        MEASURES actual usage from the response; the substrate aggregates.

        Stored on `self._last_api_usage`; the worker exposes it via /stats and the
        substrate folds it into the post-execute ResourceSample. The worker resets
        it before each execute so a failed/usage-less call can't inherit stale
        usage. Default: store-only (parallel to report_progress).
        """
        self._last_api_usage = dict(usage)

In [ ]:
#| export
# Q2 (Track 19 companion): heartbeat context manager + thread-safe report_progress.
#
# The substrate's prefetch stall-detector (Session A 2026-05-27) SIGTERMs a
# worker if the (progress, message) tuple stays unchanged for
# `SubstrateConfig.prefetch_stall_threshold_seconds`. Capabilities with quiet
# lifecycle ops (HF Hub `from_pretrained`, `torch.hub.load_state_dict_from_url`,
# whisper's urllib download) are silently at risk: their _load_model() blocks for
# 60+ seconds without emitting progress updates.
#
# Voxtral-vLLM is the first adopter (single-threaded, capability-owned poll loop —
# the heartbeat is "call report_progress on every iteration of my loop"). The 5
# remaining model-loading capabilities (Whisper / Voxtral-HF / Qwen3-FA / Demucs /
# LavaSR) have a different shape: ONE blocking call inside third-party code
# they don't own. That shape needs a BACKGROUND thread emitting heartbeats
# while the main thread blocks in `from_pretrained`.
#
# This cell ships TWO sibling primitives:
#
#   - `report_progress` thread-safety upgrade: report_progress is now safe to
#     call from any thread (the heartbeat thread + the capability's main thread
#     may both call it). Single lazy-init lock per capability instance; single-
#     threaded callers (the existing majority) get a `getattr is None` fast
#     path with zero lock overhead until the first heartbeat() call.
#
#   - `heartbeat(phase, *, interval, progress)` context manager: spawns a
#     daemon thread that re-writes `_status_message` every `interval` seconds
#     with `<base> ({elapsed:.1f}s)`. The thread NEVER touches `_progress` or
#     `_status_message_base` — those stay owned by explicit `report_progress`
#     calls. So the capability can still drive real progress fractions during the
#     block; the heartbeat just guarantees the (progress, message) tuple
#     advances even when nothing else is happening.
#
# Per Q2 Option C: this is opt-in convenience on top of the existing
# report_progress seam, NOT a forced shape. Voxtral-vLLM's capability-owned loop
# stays as-is. 5 model-loading capabilities adopt this CM in Phase 3.
def _report_progress_threadsafe(
    self,
    progress: float,  # 0.0 to 1.0, or -1.0 for indeterminate
    message: str = "",  # Descriptive status message
) -> None:
    """Thread-safe report_progress — replaces ToolCapability.report_progress.
    
    Writes `_progress` + `_status_message_base` + `_status_message` under the
    capability's `_progress_lock` if one exists (lazy-init by `heartbeat()` before
    spawning its thread). When no lock exists yet — the single-threaded common
    case before any heartbeat() call — the writes happen without lock overhead.
    
    The heartbeat thread reads `_status_message_base` (NOT `_status_message`)
    so heartbeat-amended messages don't accumulate elapsed-time suffixes when
    `report_progress` is called concurrently. The capability's call overwrites both
    fields atomically; the next heartbeat tick reads the new base.
    """
    lock = getattr(self, "_progress_lock", None)
    if lock is None:
        # Fast path for single-threaded callers (the majority — Voxtral-vLLM
        # and any capability that hasn't entered a heartbeat() block). No lock
        # overhead. heartbeat() initializes the lock before spawning its
        # thread, so concurrent multi-thread access always sees a lock.
        self._progress = progress
        self._status_message_base = message
        self._status_message = message
        return
    with lock:
        self._progress = progress
        self._status_message_base = message
        self._status_message = message


ToolCapability.report_progress = _report_progress_threadsafe


@contextmanager
def _heartbeat(
    self,
    phase: str,                    # Phase label embedded in the heartbeat message
    *,
    interval: float = 0.5,         # Seconds between heartbeats (substrate stalls at 60s default)
    progress: Optional[float] = None,  # Optional initial progress; None preserves current
) -> Iterator[None]:
    """Heartbeat context manager — spawn a daemon thread that advances the
    (progress, message) tuple every `interval` seconds, defeating the
    substrate's prefetch stall detection during silent blocking calls.
    
    Usage:
    
        def prefetch(self):
            with self.heartbeat("loading whisper model"):
                self._load_model()  # Blocking; HF Hub download / from_pretrained
    
    Behavior:
    
    - At entry: writes `(progress, phase)` to set the initial state. If
      `progress` is None, preserves whatever `self._progress` already holds
      (defaulting to 0.5 indeterminate if never set).
    - During the block: a daemon thread emits an updated `_status_message =
      "<base> ({elapsed:.1f}s)"` every `interval` seconds. The thread reads
      `_status_message_base` (set by `report_progress`) for the base, so an
      explicit `report_progress(0.3, "downloading weights")` from inside the
      block makes the next heartbeat tick display "downloading weights (Xs)".
    - At exit: signals the thread to stop, joins with timeout (the thread is
      daemon so cleanup is best-effort but reliable). The capability's final
      `_status_message` state is left as the last heartbeat-amended value;
      callers wanting a clean "completed" state should call
      `report_progress(1.0, "done")` after the with-block.
    
    Thread safety: relies on the upgraded thread-safe `report_progress`
    (loaded by this same cell). The lock is lazy-initialized HERE — before
    spawning the heartbeat thread — so concurrent main-thread + heartbeat-
    thread calls always see a lock.
    
    Cancellation: the heartbeat thread checks `stop_event` between sleeps
    and exits cleanly. If the with-block raises, the `finally` clause still
    fires `stop_event.set()`, so the thread won't leak.
    """
    # Lazy-init the lock BEFORE spawning the thread. This is the single point
    # where a write-write race is possible if multiple heartbeat() calls
    # nested or interleaved at the millisecond level — in practice capabilities
    # never call heartbeat() reentrantly, and the GIL makes the dict write
    # atomic anyway. Belt-and-suspenders: if the attribute already exists
    # (e.g. from a previous heartbeat() call in the same capability lifetime),
    # we reuse it.
    if getattr(self, "_progress_lock", None) is None:
        self._progress_lock = threading.Lock()

    # Initial state: explicit progress argument > current progress > 0.5
    # (indeterminate). The phase label becomes the message base; the thread
    # will append elapsed time on each tick.
    initial_progress = progress if progress is not None else getattr(self, "_progress", 0.5)
    self.report_progress(initial_progress, phase)

    start_time = time.time()
    stop_event = threading.Event()

    def _heartbeat_loop() -> None:
        # The thread NEVER touches _progress or _status_message_base — those
        # stay owned by explicit report_progress calls. The thread only refreshes
        # _status_message with the elapsed-time suffix. This means the substrate
        # sees the tuple advance (message changes every tick); the capability's
        # explicit progress fractions and base messages propagate cleanly.
        while not stop_event.is_set():
            elapsed = time.time() - start_time
            with self._progress_lock:
                base = getattr(self, "_status_message_base", phase) or phase
                self._status_message = f"{base} ({elapsed:.1f}s)" if base else f"({elapsed:.1f}s)"
            # Wait on the event with a timeout so we can be interrupted between
            # ticks if the with-block exits early.
            stop_event.wait(interval)

    thread = threading.Thread(target=_heartbeat_loop, daemon=True, name=f"heartbeat-{phase[:32]}")
    thread.start()
    try:
        yield
    finally:
        stop_event.set()
        # Daemon thread; join with timeout so a misbehaving thread can't block
        # the with-block from exiting. interval*2 gives the thread one full
        # cycle to notice the event; minimum 1.0s for very tight intervals.
        thread.join(timeout=max(interval * 2, 1.0))


ToolCapability.heartbeat = _heartbeat

In [ ]:
#| hide
# Q1-A + Q2 regression tests: worker-env template substitution + heartbeat CM
# + thread-safe report_progress. Run independently of any worker subprocess.

# Minimal concrete capability for these tests (cell-order independent: the
# canonical _CR4MinimalCapability is exported in a later cell, but this test cell
# can't depend on that ordering).
class _Q2TestCapability(ToolCapability):
    @property
    def name(self) -> str: return "q2-test"
    @property
    def version(self) -> str: return "0.0.0"
    def initialize(self, config=None): self._cfg = dict(config or {})
    def execute(self, *args, **kwargs): return None
    def get_config_schema(self) -> Dict[str, Any]: return {}
    def get_current_config(self) -> Dict[str, Any]: return dict(getattr(self, "_cfg", {}))


def _q1a_template_substitution_check():
    # Static default (no placeholders) passes through unchanged.
    assert expand_worker_env_template("0", {}) == "0"
    assert expand_worker_env_template("/abs/path", {"CJM_MODELS_DIR": "/foo"}) == "/abs/path"

    # Single placeholder substitutes from the mapping.
    assert expand_worker_env_template(
        "${CJM_MODELS_DIR}/huggingface",
        {"CJM_MODELS_DIR": "/srv/models", "CJM_CAPABILITY_DATA_DIR": None, "CAPABILITY_DATA_DIR": None, "CAPABILITY_NAME": "whisper"},
    ) == "/srv/models/huggingface"

    # Multiple placeholders all substituted in one pass.
    assert expand_worker_env_template(
        "${CJM_CAPABILITY_DATA_DIR}/${CAPABILITY_NAME}/nltk_data",
        {"CJM_CAPABILITY_DATA_DIR": "/home/u/.cjm", "CAPABILITY_NAME": "nltk", "CJM_MODELS_DIR": None, "CAPABILITY_DATA_DIR": None},
    ) == "/home/u/.cjm/nltk/nltk_data"

    # CAPABILITY_DATA_DIR is the convenience shorthand.
    assert expand_worker_env_template(
        "${CAPABILITY_DATA_DIR}/nltk_data",
        {"CAPABILITY_DATA_DIR": "/home/u/.cjm/data/nltk", "CAPABILITY_NAME": "nltk", "CJM_CAPABILITY_DATA_DIR": None, "CJM_MODELS_DIR": None},
    ) == "/home/u/.cjm/data/nltk/nltk_data"

    # Non-recursive: a substituted value containing $-syntax is NOT re-scanned.
    # (Defensive: ${FOO} embedded in CJM_MODELS_DIR's resolved value stays literal.)
    assert expand_worker_env_template(
        "${CJM_MODELS_DIR}/sub",
        {"CJM_MODELS_DIR": "/srv/${FOO}/abc", "CJM_CAPABILITY_DATA_DIR": None, "CAPABILITY_DATA_DIR": None, "CAPABILITY_NAME": ""},
    ) == "/srv/${FOO}/abc/sub"


def _q1a_template_unknown_placeholder_check():
    # Unknown placeholder raises CapabilityConfigError (catchable as ValueError).
    try:
        expand_worker_env_template(
            "${UNKNOWN_VAR}/path",
            {"CJM_MODELS_DIR": "/foo"},
            capability_name="whisper",
            var_name="HF_HOME",
        )
        assert False, "expected CapabilityConfigError"
    except CapabilityConfigError as e:
        # CR-5 contract: CapabilityConfigError multi-inherits ValueError.
        assert isinstance(e, ValueError)
        msg = str(e)
        assert "UNKNOWN_VAR" in msg, f"missing offending placeholder in msg: {msg}"
        assert "whisper" in msg and "HF_HOME" in msg, f"missing context in msg: {msg}"
        # fields_invalid for downstream JobError wrapping
        assert e.fields_invalid == ["HF_HOME"], f"fields_invalid: {e.fields_invalid}"


def _q1a_template_missing_value_check():
    # Allowed placeholder but unresolved value (None) raises with distinct message.
    try:
        expand_worker_env_template(
            "${CJM_MODELS_DIR}/huggingface",
            {"CJM_MODELS_DIR": None, "CJM_CAPABILITY_DATA_DIR": "/foo", "CAPABILITY_DATA_DIR": None, "CAPABILITY_NAME": ""},
            capability_name="whisper",
            var_name="HF_HOME",
        )
        assert False, "expected CapabilityConfigError"
    except CapabilityConfigError as e:
        msg = str(e)
        assert "CJM_MODELS_DIR" in msg
        assert "operator must set" in msg, f"operator-side error context missing: {msg}"


def _q1a_template_check_placeholders_validates_vocabulary():
    # template_check_placeholders (validate-time helper) returns the referenced
    # names + raises on unknowns. No need for a values map.
    assert template_check_placeholders("/abs/path") == set()
    assert template_check_placeholders("${CJM_MODELS_DIR}/foo") == {"CJM_MODELS_DIR"}
    assert template_check_placeholders(
        "${CJM_CAPABILITY_DATA_DIR}/${CAPABILITY_NAME}"
    ) == {"CJM_CAPABILITY_DATA_DIR", "CAPABILITY_NAME"}
    try:
        template_check_placeholders("${UNKNOWN_VAR}")
        assert False, "expected CapabilityConfigError"
    except CapabilityConfigError:
        pass


_q1a_template_substitution_check()
_q1a_template_unknown_placeholder_check()
_q1a_template_missing_value_check()
_q1a_template_check_placeholders_validates_vocabulary()
print("Q1-A worker-env template substitution + validation: PASS")


def _q2_report_progress_threadsafe_check():
    # Single-threaded fast-path: no lock present, writes happen directly.
    p = _Q2TestCapability()
    p.report_progress(0.3, "loading")
    assert p._progress == 0.3
    assert p._status_message == "loading"
    assert p._status_message_base == "loading"
    # No lock created in the fast path.
    assert getattr(p, "_progress_lock", None) is None

    # After heartbeat() initializes the lock, report_progress uses it.
    with p.heartbeat("phase A", interval=2.0):
        assert isinstance(p._progress_lock, type(threading.Lock()))
        old_lock = p._progress_lock
        p.report_progress(0.6, "mid-block update")
        assert p._progress == 0.6
        assert p._status_message_base == "mid-block update"
        assert p._progress_lock is old_lock, "lock identity changed mid-block"


def _q2_heartbeat_advances_tuple_check():
    # The heartbeat thread amends _status_message every interval seconds with
    # elapsed time. We verify by sampling _status_message at successive points
    # and asserting the values are distinct (substrate's stall detector would
    # see the same advance).
    p = _Q2TestCapability()
    samples = []
    with p.heartbeat("loading model", interval=0.05):
        # Sleep slightly longer than the interval to let the heartbeat fire
        # several times.
        for _ in range(4):
            time.sleep(0.06)
            samples.append(p._status_message)
    # Each sample should carry the "(N.Ns)" elapsed-time suffix, and at least
    # SOME samples must differ from each other (the tuple is advancing).
    assert all("loading model" in s for s in samples), f"base lost: {samples}"
    assert all("(" in s and "s)" in s for s in samples), f"elapsed missing: {samples}"
    assert len(set(samples)) > 1, f"tuple not advancing: {samples}"


def _q2_heartbeat_preserves_explicit_progress_check():
    # An explicit report_progress call from inside the block updates the
    # message base — the next heartbeat tick uses the NEW base, not the
    # original phase label. Verifies the "preserve explicit, append elapsed"
    # semantics.
    p = _Q2TestCapability()
    with p.heartbeat("loading model", interval=0.05):
        time.sleep(0.06)  # heartbeat ticks at least once with base="loading model"
        p.report_progress(0.7, "downloading weights")
        time.sleep(0.1)  # heartbeat ticks at least once with base="downloading weights"
        final = p._status_message
    # Final message should carry the NEW base + elapsed suffix.
    assert "downloading weights" in final, f"new base lost: {final}"
    assert "loading model" not in final, f"old base persisted: {final}"
    # _progress should also reflect the explicit call (heartbeat doesn't touch it).
    assert p._progress == 0.7


def _q2_heartbeat_thread_terminates_check():
    # The heartbeat thread must exit when the with-block exits, even if the
    # block raises. Verify by counting active threads named "heartbeat-*"
    # before vs after.
    def _heartbeat_threads_alive() -> List[threading.Thread]:
        return [t for t in threading.enumerate() if t.name.startswith("heartbeat-")]

    p = _Q2TestCapability()
    pre_count = len(_heartbeat_threads_alive())

    # Normal-exit path.
    with p.heartbeat("phase A", interval=0.05):
        time.sleep(0.06)
        # heartbeat thread should be alive here.
        assert len(_heartbeat_threads_alive()) > pre_count, "no heartbeat thread spawned"
    # Give the join() a moment + verify thread is gone.
    time.sleep(0.2)
    assert len(_heartbeat_threads_alive()) == pre_count, \
        "heartbeat thread leaked after normal exit"

    # Exception-exit path: thread must still terminate.
    try:
        with p.heartbeat("phase B", interval=0.05):
            time.sleep(0.06)
            raise RuntimeError("simulated block failure")
    except RuntimeError:
        pass
    time.sleep(0.2)
    assert len(_heartbeat_threads_alive()) == pre_count, \
        "heartbeat thread leaked after exception exit"


def _q2_report_progress_thread_safe_under_concurrency():
    # Hammer report_progress from a heartbeat thread + the main thread for
    # ~50ms; verify no exceptions and consistent state at the end. The lock
    # ensures the write triple (_progress, _status_message_base, _status_message)
    # stays consistent — without the lock, a reader could see a torn write
    # (new progress paired with old message_base).
    p = _Q2TestCapability()
    errors = []

    def writer():
        try:
            for i in range(100):
                p.report_progress(i / 100.0, f"update-{i}")
                time.sleep(0.001)
        except Exception as e:
            errors.append(e)

    with p.heartbeat("concurrent test", interval=0.001):
        writer_thread = threading.Thread(target=writer, name="writer")
        writer_thread.start()
        writer_thread.join(timeout=2.0)

    assert not errors, f"concurrent calls raised: {errors}"
    # Final state has writer's last update visible (heartbeat may have
    # re-amended _status_message, but base + progress reflect writer).
    assert p._progress == 99 / 100.0
    assert p._status_message_base == "update-99"


_q2_report_progress_threadsafe_check()
_q2_heartbeat_advances_tuple_check()
_q2_heartbeat_preserves_explicit_progress_check()
_q2_heartbeat_thread_terminates_check()
_q2_report_progress_thread_safe_under_concurrency()
print("Q2 heartbeat CM + thread-safe report_progress: PASS")

## Action Dispatcher Convention (SG-44 + SG-42)

Multi-verb capability interfaces (e.g., MediaProcessing, Graph, Text, Monitor) use a
**dispatcher-style** `execute(action, **kwargs)` rather than declaring a typed
abstract method per action. The substrate provides two helpers that let capability
authors declare action handlers cleanly:

- **`@capability_action("name")`** — marker decorator tagging a method as the handler
  for a named action.
- **`collect_capability_actions(cls)`** — walks a class (and its MRO) to derive the
  set of action names from decorated methods. Useful for declaring
  `supported_actions: ClassVar[Set[str]]` without hand-maintaining the list.

**SG-42 parameter convention:** dispatcher-style interfaces standardize on
`action=` as the dispatch parameter name. The old `command=` form used by
`cjm-infra-plugin-system` should migrate. The substrate's `@capability_action`
decorator deliberately doesn't tie itself to either name — it tags methods, and
the capability's own `execute()` decides which keyword to dispatch on. The
convention is documented; the migration is a consumer-side cascade per audit §6.

The actual migration of MediaProcessing/Graph/Text/Monitor interfaces to drop
their typed abstract methods and adopt the dispatcher + `supported_actions`
pattern lands in a later cascade step. Wave 3 ships the substrate primitives.

In [ ]:
#| export
def capability_action(
    action_name: str  # Public action name the decorated method handles
) -> Callable[[Callable], Callable]:  # Decorator
    """Marker decorator tagging a capability method as the handler for `action_name`.
    
    Sets `func._capability_action = action_name`. Capability authors with dispatcher-style
    `execute(action, **kwargs)` use `collect_capability_actions(cls)` to derive their
    `supported_actions` set from these markers rather than maintaining a separate
    list. The decorator does not change call semantics — the wrapped function is
    returned unchanged.
    """
    def decorator(func: Callable) -> Callable:
        func._capability_action = action_name
        return func
    return decorator


def collect_capability_actions(
    cls: type  # Class (or subclass) to scan for @capability_action-tagged methods
) -> Set[str]:  # Set of action names handled by `cls` (including inherited)
    """Collect action names from `@capability_action`-decorated methods on `cls`.
    
    Walks the class's MRO so subclasses inherit action handlers from base
    classes automatically. The returned set is suitable for
    `supported_actions: ClassVar[Set[str]] = collect_capability_actions(MyCapability)`
    once the capability class body has been defined.
    """
    actions: Set[str] = set()
    for ancestor in cls.__mro__:
        for attr in vars(ancestor).values():
            tag = getattr(attr, "_capability_action", None)
            if isinstance(tag, str):
                actions.add(tag)
    return actions

In [ ]:
#| export
def _dispatch_to_action(
    self,
    action: str,  # Action name to dispatch (matched against @capability_action tags)
    **kwargs,     # Forwarded verbatim to the resolved handler
) -> Any:         # Whatever the handler returns
    """T28: dispatch `action` to its `@capability_action`-tagged handler.

    Walks the instance's MRO for a method tagged `_capability_action == action`
    (the SAME markers `collect_capability_actions` / `supported_actions` are built
    from) and calls it as `handler(self, **kwargs)`. Unknown actions raise the
    typed `CapabilityInputError(fields_invalid=["action"])` (CR-5) — identical
    behaviour to the hand-rolled dispatchers this replaces.

    Dispatcher-style capabilities (MediaProcessing / Graph / Text) collapse their
    `execute` to a one-liner instead of reimplementing the MRO walk in every
    capability (the 5x copy SG-44 + this helper retire):

        @capability_action("separate_vocals")
        def _separate_vocals(self, **kwargs): ...

        supported_actions = collect_capability_actions(MyCapability)

        def execute(self, action="separate_vocals", **kwargs):
            return self.dispatch_to_action(action, **kwargs)
    """
    for klass in type(self).__mro__:
        for attr in vars(klass).values():
            if getattr(attr, "_capability_action", None) == action:
                return attr(self, **kwargs)
    raise CapabilityInputError(
        f"Unknown action: {action}", fields_invalid=["action"],
    )


ToolCapability.dispatch_to_action = _dispatch_to_action

In [ ]:
# SG-44 regression: @capability_action + collect_capability_actions form a usable
# decorator/introspection pair, and inheritance walks the MRO.
class _BaseDispatcher:
    @capability_action("hello")
    def _say_hello(self, **kwargs):
        return "hi"
    
    @capability_action("goodbye")
    def _say_goodbye(self, **kwargs):
        return "bye"


class _ExtendedDispatcher(_BaseDispatcher):
    @capability_action("wave")
    def _wave(self, **kwargs):
        return "👋"


# Decorator tags methods without changing them
assert _BaseDispatcher._say_hello._capability_action == "hello"

# Collection from base class
base_actions = collect_capability_actions(_BaseDispatcher)
assert base_actions == {"hello", "goodbye"}, f"unexpected: {base_actions}"

# Inheritance: subclass inherits parent's actions + adds its own
ext_actions = collect_capability_actions(_ExtendedDispatcher)
assert ext_actions == {"hello", "goodbye", "wave"}, f"unexpected: {ext_actions}"

# Undecorated methods don't contribute
class _Plain:
    def regular(self): return None
assert collect_capability_actions(_Plain) == set()

# Methods still callable after decoration
inst = _BaseDispatcher()
assert inst._say_hello() == "hi"

print("SG-44 @capability_action + collect_capability_actions: PASS")

In [ ]:
# T28 regression: ToolCapability.dispatch_to_action routes to the
# @capability_action handler via the MRO walk + forwards kwargs; unknown actions
# raise the typed CapabilityInputError(fields_invalid=["action"]). This is the
# substrate helper the 5 dispatcher capabilities (Demucs/LavaSR/NLTK/FFmpeg/
# SQLite-graph) call instead of reimplementing the loop.
class _T28DispatchCapability(ToolCapability):
    @property
    def name(self) -> str: return "t28-dispatch"
    @property
    def version(self) -> str: return "0.0.0"
    def initialize(self, config=None): pass
    def get_config_schema(self) -> Dict[str, Any]: return {}
    def get_current_config(self) -> Dict[str, Any]: return {}
    def execute(self, action: str = "ping", **kwargs):
        return self.dispatch_to_action(action, **kwargs)

    @capability_action("ping")
    def _ping(self, **kwargs): return "pong"

    @capability_action("echo")
    def _echo(self, value=None, **kwargs): return value


# Inheritance: a subclass inherits the base's @capability_action handlers (MRO walk).
class _T28ExtendedCapability(_T28DispatchCapability):
    @capability_action("shout")
    def _shout(self, text="", **kwargs): return text.upper()


_t28 = _T28DispatchCapability()
# Routes to the tagged handler + forwards kwargs
assert _t28.execute("ping") == "pong"
assert _t28.dispatch_to_action("echo", value=42) == 42
# supported_actions (collect_capability_actions) and dispatch share the same markers
assert collect_capability_actions(_T28DispatchCapability) == {"ping", "echo"}
# Unknown action raises typed CapabilityInputError with fields_invalid=["action"]
try:
    _t28.dispatch_to_action("nope")
    assert False, "expected CapabilityInputError"
except CapabilityInputError as e:
    assert e.fields_invalid == ["action"], f"unexpected fields_invalid: {e.fields_invalid}"
    assert isinstance(e, ValueError), "CapabilityInputError must multi-inherit ValueError (CR-5)"

# MRO walk: subclass dispatches BOTH its own and inherited handlers
_t28e = _T28ExtendedCapability()
assert _t28e.dispatch_to_action("shout", text="hi") == "HI"
assert _t28e.dispatch_to_action("ping") == "pong"
assert collect_capability_actions(_T28ExtendedCapability) == {"ping", "echo", "shout"}

print("T28 dispatch_to_action: PASS")

The interface provides:

- **Identity**: `name` and `version` properties for capability identification
- **Lifecycle**: `initialize()` / `prefetch()` / `cleanup()` + `on_disable()` / `on_enable()` signal hooks
- **Configuration**: `get_config_schema()` / `get_current_config()` / `get_config_options()` / `reconfigure()` (+ `RELOAD_TRIGGER` declarative releases)
- **Cancellation**: `cancel()` / `check_cancel()` / `register_cancel_callback()` / `cancel_signal_to()` cooperative primitives
- **Observability**: `report_progress()` / `report_usage()` / `heartbeat()`
- **Multi-op dispatch**: `dispatch_to_action()` for genuinely multi-op tools (ffmpeg, graph storage)

The task channel (`execute` / typed task methods) belongs to task adapters,
not this interface; `execute_stream`'s transitional default remains for
fused-era capabilities only.

## Example: Implementing a Capability

Here's a complete example showing how to implement a concrete capability:

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class ExampleConfig:
    """Configuration for the example capability."""
    mode: str = "balanced"
    threshold: float = 0.5
    max_workers: int = 4

class ExampleCapability(ToolCapability):
    """A simple example capability implementation."""

    def __init__(self):
        self._config: ExampleConfig = ExampleConfig()
        self._resource: Optional[str] = None

    @property
    def name(self) -> str:
        return "example-capability"
    
    @property
    def version(self) -> str:
        return "1.0.0"
    
    def initialize(self, config: Optional[Dict[str, Any]] = None) -> None:
        """Initialize or re-configure the capability."""
        if config is None:
            config = {}
        
        # Merge with defaults
        current = asdict(self._config)
        current.update(config)
        self._config = ExampleConfig(**current)
        
        # Initialize resources based on config
        self._resource = f"Resource-{self._config.mode}"

    def execute(self, input_data: str, **kwargs) -> str:
        """Process input data."""
        return f"Processed '{input_data}' using {self._resource}"

    def get_config_schema(self) -> Dict[str, Any]:
        """Return JSON Schema for configuration."""
        return {
            "type": "object",
            "properties": {
                "mode": {
                    "type": "string",
                    "enum": ["fast", "balanced", "quality"],
                    "default": "balanced"
                },
                "threshold": {
                    "type": "number",
                    "minimum": 0.0,
                    "maximum": 1.0,
                    "default": 0.5
                },
                "max_workers": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 16,
                    "default": 4
                }
            }
        }

    def get_current_config(self) -> Dict[str, Any]:
        """Return current configuration."""
        return asdict(self._config)

    def cleanup(self) -> None:
        """Clean up resources."""
        self._resource = None

In [ ]:
# Test the example capability
capability = ExampleCapability()
capability.initialize()

print(f"Capability: {capability.name} v{capability.version}")
print(f"\nConfig Schema:")
print(capability.get_config_schema())
print(f"\nCurrent Config:")
print(capability.get_current_config())

In [ ]:
# Test execution
result = capability.execute("sample_data")
print(f"Result: {result}")

# Test re-initialization with new config
capability.initialize({"mode": "quality", "threshold": 0.8})
print(f"\nAfter re-init config: {capability.get_current_config()}")
result = capability.execute("more_data")
print(f"Result: {result}")

# Test cleanup
capability.cleanup()
print(f"\nAfter cleanup, resource is cleared")

In [ ]:
# Test streaming (default implementation yields single result)
capability.initialize({"mode": "balanced"})

print("Streaming execution:")
for chunk in capability.execute_stream("stream_data"):
    print(f"  Chunk: {chunk}")

## CR-4 regression tests

Exercise the lifecycle hooks + helpers + cancellation primitives independently of any
worker subprocess. The tests use bare `ToolCapability` subclasses without going
through `RemoteCapabilityProxy`.

What's covered:

- `cleanup()` made optional — a subclass with no `cleanup` override is instantiable.
- `prefetch()` default no-op.
- `fields_that_changed` symmetric difference + asymmetric-presence handling.
- `reconfigure_with_triggers` walks `RELOAD_TRIGGER` metadata, dispatches `_release_<trigger>`,
  de-dupes shared triggers, silently skips capabilities without `config_class`, and survives
  a misbehaving `_release` method.
- `cancel()` sets `_cancel_requested` + fires registered callbacks (logging and skipping
  misbehaving callbacks).
- `check_cancel()` raises `CapabilityCancelledError` only when flag is set.
- `cancel_signal_to` context manager registers + deregisters cleanly even when the
  with-block raises.

In [ ]:
#| export
class _CR4MinimalCapability(ToolCapability):
    """Concrete capability satisfying abstracts; relies on CR-4 default cleanup()."""
    @property
    def name(self) -> str: return "cr4-minimal"
    @property
    def version(self) -> str: return "0.0.0"
    def initialize(self, config=None): self._cfg = dict(config or {})
    def execute(self, *args, **kwargs): return None
    def get_config_schema(self) -> Dict[str, Any]: return {}
    def get_current_config(self) -> Dict[str, Any]: return dict(getattr(self, "_cfg", {}))

In [ ]:
#| hide
# CR-4 test scaffolding: a minimal concrete capability that satisfies all required
# abstracts and intentionally does NOT override cleanup. If cleanup were still
# @abstractmethod, this class would fail to instantiate (ABCMeta error). With
# cleanup made optional in CR-4, it works.

class _CR4MinimalCapability(ToolCapability):
    """Concrete capability satisfying abstracts; relies on CR-4 default cleanup()."""
    @property
    def name(self) -> str: return "cr4-minimal"
    @property
    def version(self) -> str: return "0.0.0"
    def initialize(self, config=None): self._cfg = dict(config or {})
    def execute(self, *args, **kwargs): return None
    def get_config_schema(self) -> Dict[str, Any]: return {}
    def get_current_config(self) -> Dict[str, Any]: return dict(getattr(self, "_cfg", {}))


def _cr4_cleanup_optional_check():
    # CR-4: cleanup no longer @abstractmethod — capability without override is instantiable
    p = _CR4MinimalCapability()
    p.cleanup()  # No-op default; must not raise
    p.prefetch()  # No-op default; must not raise

_cr4_cleanup_optional_check()


def _cr4_fields_that_changed_check():
    p = _CR4MinimalCapability()
    
    # Same dict: no changes
    assert p.fields_that_changed({"a": 1, "b": 2}, {"a": 1, "b": 2}) == set()
    
    # Value change
    assert p.fields_that_changed({"a": 1}, {"a": 2}) == {"a"}
    
    # Key only in old: counts as change (treated as new=None)
    assert p.fields_that_changed({"a": 1}, {}) == {"a"}
    
    # Key only in new: counts as change (treated as old=None)
    assert p.fields_that_changed({}, {"a": 1}) == {"a"}
    
    # Asymmetric union of keys
    assert p.fields_that_changed({"a": 1, "b": 2}, {"b": 2, "c": 3}) == {"a", "c"}
    
    # Nested-structure equality is by value
    assert p.fields_that_changed({"x": [1, 2]}, {"x": [1, 2]}) == set()
    assert p.fields_that_changed({"x": [1, 2]}, {"x": [2, 1]}) == {"x"}

_cr4_fields_that_changed_check()

In [ ]:
#| hide
# CR-4 reconfigure_with_triggers: walks RELOAD_TRIGGER metadata, fires
# _release_<trigger> for fields whose values changed, de-dupes shared triggers,
# silently skips capabilities without config_class, survives a raising release.
from dataclasses import dataclass, field as _field


@dataclass
class _CR4WhisperConfig:
    """Test config dataclass with two RELOAD_TRIGGER-tagged fields sharing a trigger."""
    model: str = _field(default="base", metadata={RELOAD_TRIGGER: "model"})
    revision: str = _field(default="main", metadata={RELOAD_TRIGGER: "model"})  # SAME trigger
    device: str = _field(default="cuda", metadata={RELOAD_TRIGGER: "device"})  # DIFFERENT trigger
    temperature: float = _field(default=0.0)  # NO trigger — change shouldn't fire release


class _CR4TriggerCapability(_CR4MinimalCapability):
    """Capability that opts into the declarative RELOAD_TRIGGER pattern."""
    config_class = _CR4WhisperConfig
    
    def __init__(self):
        self.released = []  # Tracks which triggers fired, in order
    
    def _release_model(self):
        self.released.append("model")
    
    def _release_device(self):
        self.released.append("device")


class _CR4RaisingTriggerCapability(_CR4TriggerCapability):
    """Variant where _release_model raises — verifies other triggers still fire."""
    def _release_model(self):
        self.released.append("model-attempted")
        raise RuntimeError("simulated release failure")


def _cr4_reconfigure_with_triggers_check():
    # Case 1: no change → no triggers fire
    p = _CR4TriggerCapability()
    p.reconfigure_with_triggers({"model": "base"}, {"model": "base"})
    assert p.released == [], f"no-change run fired: {p.released}"
    
    # Case 2: only `model` changed → fires _release_model exactly once
    p = _CR4TriggerCapability()
    p.reconfigure_with_triggers({"model": "base"}, {"model": "large"})
    assert p.released == ["model"], f"unexpected: {p.released}"
    
    # Case 3: model AND revision both changed (same trigger) → fires _release_model ONCE
    # (the de-duping invariant: shared trigger means one release, not two).
    p = _CR4TriggerCapability()
    p.reconfigure_with_triggers(
        {"model": "base", "revision": "main"},
        {"model": "large", "revision": "v2"},
    )
    assert p.released == ["model"], f"de-dupe broken: {p.released}"
    
    # Case 4: device + model changed → both triggers fire (order is set-iteration; assert
    # contents not sequence). De-dupe still applies to revision/model.
    p = _CR4TriggerCapability()
    p.reconfigure_with_triggers(
        {"model": "base", "device": "cuda"},
        {"model": "large", "device": "cpu"},
    )
    assert set(p.released) == {"model", "device"}, f"unexpected: {p.released}"
    
    # Case 5: only `temperature` (no RELOAD_TRIGGER) changed → no release fires
    p = _CR4TriggerCapability()
    p.reconfigure_with_triggers({"temperature": 0.0}, {"temperature": 0.7})
    assert p.released == [], f"non-triggered field fired release: {p.released}"
    
    # Case 6: raising _release_X is logged + skipped; other triggers still fire
    p = _CR4RaisingTriggerCapability()
    p.reconfigure_with_triggers(
        {"model": "base", "device": "cuda"},
        {"model": "large", "device": "cpu"},
    )
    # model attempt was made (then raised); device still fired
    assert "model-attempted" in p.released
    assert "device" in p.released, "device trigger must fire even after model's raise"
    
    # Case 7: capability without config_class → silent no-op (no error, no release)
    plain = _CR4MinimalCapability()
    plain.reconfigure_with_triggers({"x": 1}, {"x": 2})  # Must not raise

_cr4_reconfigure_with_triggers_check()


def _cr4_reconfigure_delegates_check():
    # Default reconfigure(old, new) delegates to reconfigure_with_triggers.
    # Stub the helper to verify the delegation contract.
    p = _CR4TriggerCapability()
    p.reconfigure({"model": "base"}, {"model": "large"})
    assert p.released == ["model"], f"reconfigure() must delegate: {p.released}"
    
    # None args are tolerated by reconfigure (defaults to {})
    p2 = _CR4TriggerCapability()
    p2.reconfigure(None, {"model": "large"})
    # old={} → {"model":"large"} differs from missing key → trigger fires
    assert p2.released == ["model"]

_cr4_reconfigure_delegates_check()

In [ ]:
#| hide
# CR-4 completion (2026-05-25): reconfigure is now the substrate's canonical
# delta path (CapabilityManager.update_capability_config routes here, not initialize).
# After firing RELOAD_TRIGGER releases it RE-APPLIES config: it prefers an
# `_apply_config(config)` seam (clean init-once / reconfigure-delta split) and
# falls back to a full `initialize(config)` otherwise. These tests lock the
# two-phase contract: (1) release fires BEFORE re-apply, (2) config is applied,
# (3) initialize-fallback when no _apply_config.

class _CR4ApplyConfigCapability(_CR4TriggerCapability):
    """Capability with the clean _apply_config seam."""
    def __init__(self):
        super().__init__()
        self.events = []          # ordered phase log
        self._cfg = {}
    def _release_model(self):
        super()._release_model()
        self.events.append("release:model")
    def _apply_config(self, config):
        self.events.append(("apply", dict(config)))
        self._cfg = dict(config)
    def initialize(self, config=None):
        # must NOT be invoked by reconfigure when _apply_config is present
        self.events.append("initialize")
        self._cfg = dict(config or {})


class _CR4FallbackCapability(_CR4TriggerCapability):
    """Capability WITHOUT _apply_config — reconfigure must fall back to initialize(new)."""
    def __init__(self):
        super().__init__()
        self.events = []
        self._cfg = {}
    def _release_model(self):
        super()._release_model()
        self.events.append("release:model")
    def initialize(self, config=None):
        self.events.append(("initialize", dict(config or {})))
        self._cfg = dict(config or {})


def _cr4_reconfigure_completion_check():
    # _apply_config path: release fires THEN apply; initialize() not used; config applied
    p = _CR4ApplyConfigCapability()
    p.reconfigure({"model": "base"}, {"model": "large"})
    assert p.events == ["release:model", ("apply", {"model": "large"})], p.events
    assert p.get_current_config() == {"model": "large"}
    assert "initialize" not in p.events

    # fallback path: no _apply_config -> release fires THEN initialize(new) applies config
    p = _CR4FallbackCapability()
    p.reconfigure({"model": "base"}, {"model": "large"})
    assert p.events == ["release:model", ("initialize", {"model": "large"})], p.events
    assert p.get_current_config() == {"model": "large"}

    # non-trigger field change still applies config (no release fired)
    p = _CR4FallbackCapability()
    p.reconfigure({"temperature": 0.0}, {"temperature": 0.7})
    assert p.events == [("initialize", {"temperature": 0.7})], p.events
    assert p.get_current_config() == {"temperature": 0.7}

    # None old_config tolerated; config still applied
    p = _CR4ApplyConfigCapability()
    p.reconfigure(None, {"model": "large"})
    assert p.get_current_config() == {"model": "large"}

_cr4_reconfigure_completion_check()


In [ ]:
#| hide
# CR-11: get_config_options default is {} (static capabilities); capabilities override to
# surface live option domains. Dataclasses must be asdict-serializable for the
# worker's EnhancedJSONEncoder round-trip.
import dataclasses as _dc

def _cr11_default_is_empty():
    assert _CR4MinimalCapability().get_config_options() == {}


class _CR11DynamicCapability(_CR4MinimalCapability):
    def get_config_options(self):
        return {
            "model": FieldOptions(
                options=[
                    ConfigOption("gemini-2.5-flash", "Gemini 2.5 Flash",
                                 {"input_token_limit": 1048576, "output_token_limit": 65536}),
                    ConfigOption("gemini-2.5-pro", "Gemini 2.5 Pro",
                                 {"input_token_limit": 1048576, "output_token_limit": 65536}),
                ],
                constraints={"max_output_tokens": {"max": 65536}},
            )
        }


def _cr11_override_and_serialize():
    opts = _CR11DynamicCapability().get_config_options()
    assert set(opts) == {"model"}
    fo = opts["model"]
    assert isinstance(fo, FieldOptions)
    assert [o.value for o in fo.options] == ["gemini-2.5-flash", "gemini-2.5-pro"]
    assert fo.options[0].metadata["output_token_limit"] == 65536
    # asdict round-trip mirrors the worker's EnhancedJSONEncoder serialization
    d = {k: _dc.asdict(v) for k, v in opts.items()}
    assert d["model"]["options"][1]["label"] == "Gemini 2.5 Pro"
    assert d["model"]["constraints"]["max_output_tokens"]["max"] == 65536

_cr11_default_is_empty()
_cr11_override_and_serialize()


In [ ]:
#| hide
# CR-12: EnvVarSpec is the worker-environment contract entry. Verify the two
# flavors (secret / visible) + asdict round-trip (the manifest stores WORKER_ENV
# as a list of dicts via the install-time introspection script).
import dataclasses as _dc12

def _envvarspec_check():
    # Secret flavor: no default (a baked-in secret is a leak).
    secret = EnvVarSpec(name="GEMINI_API_KEY", secret=True, required=True,
                        label="Gemini API Key", description="Google Gemini API key")
    assert secret.secret is True and secret.required is True
    assert secret.default is None, "secret must not carry a default value"

    # Visible flavor: carries a default, resolved via the override chain.
    visible = EnvVarSpec(name="CUDA_VISIBLE_DEVICES", default="0",
                         label="GPU Device", description="Which GPU index the worker uses")
    assert visible.secret is False and visible.default == "0"

    # A capability declares its contract as a WORKER_ENV list. asdict round-trips so
    # the introspection script can JSON-emit it into the manifest code section.
    class _APICapability(_CR4MinimalCapability):
        WORKER_ENV = [secret, visible]

    dicts = [_dc12.asdict(s) for s in _APICapability.WORKER_ENV]
    assert dicts[0]["name"] == "GEMINI_API_KEY" and dicts[0]["secret"] is True
    assert dicts[1]["name"] == "CUDA_VISIBLE_DEVICES" and dicts[1]["default"] == "0"

    # Capabilities without a WORKER_ENV declaration are the norm (no spawn-env contract).
    assert getattr(_CR4MinimalCapability, "WORKER_ENV", None) is None

_envvarspec_check()
print("CR-12 EnvVarSpec worker-env contract: PASS")


In [ ]:
#| hide
# CR-4 cancellation primitives: cancel() sets flag + fires callbacks;
# check_cancel() raises CapabilityCancelledError when flag set; cancel_signal_to
# context manager registers/deregisters across normal and exception paths.

def _cr4_cancellation_check():
    # Fresh instance: flag is False (class-level default), check_cancel doesn't raise
    p = _CR4MinimalCapability()
    assert p._cancel_requested is False
    p.check_cancel()  # No-op
    
    # cancel() sets the flag
    p.cancel()
    assert p._cancel_requested is True
    
    # check_cancel() now raises CapabilityCancelledError
    try:
        p.check_cancel()
        assert False, "expected CapabilityCancelledError"
    except CapabilityCancelledError as e:
        assert e.capability_name == "cr4-minimal"
        # CapabilityCancelledError must NOT be ValueError-catchable per CR-5 discipline
        assert not isinstance(e, ValueError)
    
    # Re-raising is idempotent — flag stays set after raise (substrate resets it
    # between executions; we don't auto-reset on raise to avoid masking re-checks).
    try:
        p.check_cancel()
        assert False, "expected second CapabilityCancelledError"
    except CapabilityCancelledError:
        pass


def _cr4_cancel_callbacks_check():
    p = _CR4MinimalCapability()
    calls = []
    
    def cb1(): calls.append("cb1")
    def cb2(): calls.append("cb2")
    
    p.register_cancel_callback(cb1)
    p.register_cancel_callback(cb2)
    
    # Callbacks fire in registration order when cancel() is called
    p.cancel()
    assert calls == ["cb1", "cb2"], f"order broken: {calls}"
    
    # Subsequent cancel() fires callbacks again (substrate may signal multiple
    # times during shutdown; idempotency is the callback author's responsibility,
    # but the substrate side fires every time).
    p.cancel()
    assert calls == ["cb1", "cb2", "cb1", "cb2"]


def _cr4_cancel_callback_misbehaving_check():
    p = _CR4MinimalCapability()
    calls = []
    
    def good_cb(): calls.append("good")
    def bad_cb(): raise RuntimeError("misbehaving callback")
    def later_cb(): calls.append("later")
    
    p.register_cancel_callback(good_cb)
    p.register_cancel_callback(bad_cb)
    p.register_cancel_callback(later_cb)
    
    # Misbehaving callback is logged and skipped; subsequent callbacks still fire
    p.cancel()
    assert calls == ["good", "later"], f"misbehaving callback blocked others: {calls}"


def _cr4_cancel_signal_to_check():
    p = _CR4MinimalCapability()
    calls = []
    
    def teardown_cb(): calls.append("teardown")
    
    # Normal path: callback registered, fires on cancel inside block, deregistered
    # after block.
    with p.cancel_signal_to(teardown_cb):
        # Callback is registered while we're inside the with-block
        assert teardown_cb in p._cancel_callbacks
        p.cancel()
        assert calls == ["teardown"]
    # After with-block exits, callback is deregistered
    assert teardown_cb not in p._cancel_callbacks
    
    # Re-cancel after block: callback no longer fires
    p.cancel()
    assert calls == ["teardown"], f"deregistered callback fired again: {calls}"
    
    # Exception path: callback deregistered even if with-block raises
    p2 = _CR4MinimalCapability()
    teardown2 = lambda: None
    try:
        with p2.cancel_signal_to(teardown2):
            assert teardown2 in p2._cancel_callbacks
            raise ValueError("scoped failure")
    except ValueError:
        pass
    assert teardown2 not in p2._cancel_callbacks, \
        "cancel_signal_to must deregister even when the with-block raises"


_cr4_cancellation_check()
_cr4_cancel_callbacks_check()
_cr4_cancel_callback_misbehaving_check()
_cr4_cancel_signal_to_check()
print("CR-4 lifecycle hooks + helpers + cancellation primitives: PASS")

## Structural surface derivation

Surface-based, adapter-driven compatibility (pass-2 Thread 3): the
tool-capability manifest records the class's **structural surface**
(public methods + signatures, properties, class attributes) by pure
self-introspection — the capability has ZERO knowledge of protocols or
adapters. The adapter declares `required_tool_protocol`; the substrate
matches the protocol against recorded surfaces host-side, against
UNLOADED capabilities (the matching machinery is CR-17 pt 2, stage 4).

Two call sites use this one derivation: the `_generate_manifest`
introspection script (runs in the capability's env at install/regenerate
time → `code.structural_surface` + a witness hash) and the worker's
`/structural_surface` endpoint (the live companion the manager compares
against the hash → `surface_drift` flag; third instance of the CR-8
hashed-witness + live-companion idiom).

In [ ]:
#| export
def derive_structural_surface(
    cls: type,  # The capability class to introspect
) -> Dict[str, Any]:  # {"methods": [...], "properties": [...], "attributes": [...]}
    """Record a capability class's structural surface by pure self-introspection.

    The FULL public surface is recorded, inherited members included —
    the surface is what adapter protocols match against, and protocol
    members may name inherited methods. Deterministic (name-sorted) so
    the canonical-JSON witness hash is stable across runs.

    Classification: `property` → properties (names only); functions
    (static/class methods unwrapped) → methods with `str(inspect.signature)`
    + the parameter NAME list `params` (self excluded — the CR-17 pt 2
    surface matcher's input; stage 4);
    everything else public → attributes with the value's type name
    (config_class, supported_actions, WORKER_ENV, ...).
    """
    methods: List[Dict[str, str]] = []
    properties: List[str] = []
    attributes: List[Dict[str, str]] = []
    for name in sorted(dir(cls)):
        if name.startswith("_"):
            continue
        attr = inspect.getattr_static(cls, name)
        if isinstance(attr, property):
            properties.append(name)
            continue
        if isinstance(attr, (staticmethod, classmethod)):
            attr = attr.__func__
        if inspect.isfunction(attr):
            try:
                signature = inspect.signature(attr)
                sig = str(signature)
                params = [p for p in signature.parameters if p != "self"]
            except (ValueError, TypeError):
                sig, params = "(...)", []
            methods.append({"name": name, "signature": sig, "params": params})
        else:
            attributes.append({"name": name, "type": type(attr).__name__})
    return {"methods": methods, "properties": properties, "attributes": attributes}

In [ ]:
# derive_structural_surface: full public surface (inherited base methods
# included — they ARE surface), deterministic ordering, classification.
surf = derive_structural_surface(_CR4MinimalCapability)
names = {m["name"] for m in surf["methods"]}
assert "execute" in names              # the fused-era capability defines it
assert "reconfigure" in names          # inherited ToolCapability method
assert "dispatch_to_action" in names   # patched-on members are surface too
assert "name" in surf["properties"] and "version" in surf["properties"]
sig = next(m["signature"] for m in surf["methods"] if m["name"] == "initialize")
assert "config" in sig
assert surf == derive_structural_surface(_CR4MinimalCapability)  # deterministic
assert [m["name"] for m in surf["methods"]] == sorted(m["name"] for m in surf["methods"])
print(f"derive_structural_surface OK: {len(surf['methods'])} methods, "
      f"{len(surf['properties'])} properties, {len(surf['attributes'])} attributes")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()